# 项目三：带有短期记忆的客服 — Memory（记忆系统）

在前两个项目中，我们已经学会了：
- 项目一：调用 DeepSeek API，用 Prompt 控制模型输出
- 项目二：让大模型调用外部工具（Tool Use）

但你有没有发现一个问题——**每次调用 API，模型都是"失忆"的**。

> 你问它"我叫什么名字？"，它说不知道。你告诉它"我叫小明"，下一轮再问，它又忘了。

**因为大模型本身没有记忆。** 每次 API 调用都是独立的，模型不会自动记住上次聊了什么。

本项目将解决这个问题。你将学习：

1. 理解智能体的 **短期记忆（Short-term Memory）** 机制
2. 掌握通过 `messages` 列表维护对话上下文的方法
3. 学会 **滑动窗口** 和 **摘要压缩** 两种记忆管理策略
4. 构建一个能"记住"之前对话内容的客服智能体

---

## 一、什么是 Memory？

### 1.1 为什么大模型会"失忆"？

大模型的 API 是**无状态（Stateless）**的。每次调用就像和一个完全陌生的人对话——

```
第 1 次调用：你告诉它 "我叫小明"
第 2 次调用：你问 "我叫什么？" → 它完全不知道，因为第 1 次的对话已经"消失"了
```

这就像你每天醒来都会失去前一天的记忆（电影《初恋50次》的情节）。

### 1.2 解决方案：把历史对话"喂"回去

既然模型没有记忆，我们就**主动把之前的对话记录告诉它**：

```
第 1 次调用：messages = [
    {"role": "user", "content": "我叫小明"}
]

第 2 次调用：messages = [
    {"role": "user", "content": "我叫小明"},           ← 历史记录
    {"role": "assistant", "content": "你好小明！"},    ← 历史记录
    {"role": "user", "content": "我叫什么名字？"}      ← 当前问题
]
→ 模型看到历史记录，就能回答"你叫小明"
```

**这就是短期记忆的本质：把对话历史存在一个列表里，每次调用时把完整的历史一起发给模型。**

### 1.3 记忆的两种类型

| 类型 | 说明 | 类比 | 实现方式 |
|------|------|------|----------|
| **短期记忆** | 当前对话的上下文 | 你正在和朋友聊天，记得刚才说了什么 | 把 messages 列表传给 API |
| **长期记忆** | 跨会话的持久知识 | 你记得去年认识的朋友的名字 | 存入数据库/文件，下次会话时读取 |

本项目聚焦于**短期记忆**——让智能体在一次对话中"记住"上下文。

### 1.4 记忆管理的挑战

直接把所有历史都塞给模型，会遇到一个问题：**上下文窗口有限**。

```
模型最多能处理 4096 个 token（大约 3000 个汉字）
如果聊了 50 轮，历史记录可能超过这个限制
```

所以我们需要**记忆管理策略**：

| 策略 | 思路 | 优点 | 缺点 |
|------|------|------|------|
| **滑动窗口** | 只保留最近 N 轮对话 | 简单高效 | 会丢失早期信息 |
| **摘要压缩** | 把早期对话总结成一段话 | 保留关键信息，节省 token | 总结可能丢失细节 |

---

## 二、基础准备

### 2.1 导入依赖

本项目继续使用 DeepSeek API，沿用前两个项目的基础设施：

In [ ]:
# Ensure openai is installed in JupyterLite/Pyodide before importing it.
try:
    from openai import OpenAI
except ModuleNotFoundError:
    import micropip
    await micropip.install("openai")
    from openai import OpenAI

import os

# 初始化 DeepSeek 客户端
api_key = os.environ.get("DEEPSEEK_API_KEY", "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

def chat(messages, temperature=0.7):
    """调用 DeepSeek API 的通用函数"""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print("DeepSeek API 连接成功！")

### 2.2 回顾 messages 参数

在项目一和项目二中，我们已经接触过 `messages` 参数。这里再系统回顾一下：

```python
messages = [
    {"role": "system",    "content": "你是一个客服助手"},    # 系统指令（可选，放在最前面）
    {"role": "user",      "content": "你好"},               # 用户消息
    {"role": "assistant", "content": "你好！有什么可以帮您？"}, # 助手回复
    {"role": "user",      "content": "我想退货"},            # 用户继续提问
]
```

**关键点：**

| 角色 | 作用 | 说明 |
|------|------|------|
| `system` | 系统指令 | 定义助手的行为，放在列表最前面 |
| `user` | 用户消息 | 用户说的话 |
| `assistant` | 助手回复 | 模型之前说的话 |

**记忆的核心就是：把之前的 user 和 assistant 消息保留在 messages 列表中，一起发给模型。**

---

## 三、示例一：最基础的记忆 —— 对话历史拼接

我们先实现最简单的记忆方式：**用一个列表存储所有对话历史，每次调用时全部发给模型。**

### 3.1 思路

```
┌─────────────────────────────────────────────────────────┐
│  memory = []  （空列表，初始状态）                        │
│                                                          │
│  第 1 轮：                                               │
│    用户说 "我叫小明"                                      │
│    → 加入 memory: ["User: 我叫小明"]                      │
│    → 发给模型，得到回复 "你好小明！"                       │
│    → 加入 memory: ["User: 我叫小明",                      │
│                    "Agent: 你好小明！"]                    │
│                                                          │
│  第 2 轮：                                               │
│    用户说 "我叫什么？"                                    │
│    → 加入 memory: [..., "User: 我叫什么？"]               │
│    → 把完整 memory 发给模型                               │
│    → 模型看到历史记录，回答 "你叫小明"                     │
│    → 加入 memory: [..., "Agent: 你叫小明"]                │
└─────────────────────────────────────────────────────────┘
```

### 3.2 代码实现

In [ ]:
class SimpleMemoryAgent:
    """最简单的记忆智能体：把所有对话历史都记住"""
    
    def __init__(self, system_prompt="你是一个友好的助手。"):
        # 用 messages 列表存储对话历史
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def chat(self, user_text):
        """处理一轮对话"""
        # 第 1 步：把用户消息加入历史
        self.messages.append({"role": "user", "content": user_text})
        
        # 第 2 步：把完整历史发给模型
        response = chat(self.messages)
        
        # 第 3 步：把模型的回复也加入历史
        self.messages.append({"role": "assistant", "content": response})
        
        return response
    
    def show_memory(self):
        """查看当前记忆内容"""
        print(f"当前记忆中有 {len(self.messages)} 条消息：")
        for i, msg in enumerate(self.messages):
            role = msg["role"]
            content = msg["content"][:50]  # 只显示前50个字符
            print(f"  [{i}] {role}: {content}")


# ===== 测试 =====
agent = SimpleMemoryAgent("你是一个友好的中文助手，请用简短的话回答。")

print("=== 第 1 轮 ===")
reply1 = agent.chat("我叫小明，我是一名大学生。")
print(f"助手: {reply1}")

print("\n=== 第 2 轮 ===")
reply2 = agent.chat("我叫什么名字？我是做什么的？")
print(f"助手: {reply2}")

print("\n=== 第 3 轮 ===")
reply3 = agent.chat("帮我推荐一道适合学生做的简单菜")
print(f"助手: {reply3}")

print("\n=== 查看记忆 ===")
agent.show_memory()

### 3.3 运行分析

运行上面的代码，你会发现：

- **第 1 轮**：模型正常回答
- **第 2 轮**：模型能记住"你叫小明，是大学生"——因为历史记录里有
- **第 3 轮**：模型推荐菜品时，可能会考虑"你是学生"这个背景

**查看记忆**时，你会看到 messages 列表里有 7 条消息（1 条 system + 3 轮 × 2 条）。

### 3.4 这种方式的问题

如果聊了 100 轮，messages 列表就会非常长，可能：
1. **超出模型的上下文限制**（deepseek-chat 最大 64K token）
2. **API 调用变慢**（发送的数据量太大）
3. **API 费用增加**（token 越多越贵）

所以我们需要更好的记忆管理策略。接下来看示例二。

---

## 四、示例二：滑动窗口 + 摘要压缩

### 4.1 策略说明

我们结合两种策略来管理记忆：

```
┌──────────────────────────────────────────────────────────────┐
│  messages = [                                                 │
│      system_prompt,        ← 始终保留                        │
│      summary,              ← 早期对话的摘要（可能为空）       │
│      最近 N 轮对话,         ← 滑动窗口                       │
│  ]                                                            │
│                                                               │
│  当对话轮数超过窗口大小时：                                    │
│    1. 把最早的几轮对话发给模型，让它生成摘要                    │
│    2. 用摘要替换掉那些早期对话                                 │
│    3. 只保留最近 N 轮 + 摘要                                  │
└──────────────────────────────────────────────────────────────┘
```

### 4.2 代码实现

In [ ]:
class SmartMemoryAgent:
    """带滑动窗口 + 摘要压缩的记忆智能体"""
    
    def __init__(self, system_prompt="你是一个友好的客服助手。", window_size=4):
        self.system_prompt = system_prompt
        self.window_size = window_size  # 保留最近几轮对话
        self.summary = ""              # 早期对话的摘要
        self.recent_messages = []      # 最近 N 轮对话
    
    def _summarize(self, old_messages):
        """把旧对话压缩成摘要"""
        # 把旧对话拼成文本
        old_text = "\n".join(
            f"{m['role']}: {m['content']}" for m in old_messages
        )
        
        # 让模型生成摘要
        summary_prompt = [
            {"role": "system", "content": "你是一个摘要助手。请用 1-2 句话总结以下对话的关键信息。"},
            {"role": "user", "content": f"请总结以下对话：\n{old_text}"}
        ]
        return chat(summary_prompt, temperature=0.3)  # 低温度，更稳定
    
    def _build_messages(self):
        """构建发给模型的完整 messages 列表"""
        messages = [{"role": "system", "content": self.system_prompt}]
        
        # 如果有摘要，加入摘要
        if self.summary:
            messages.append({
                "role": "system", 
                "content": f"以下是之前对话的摘要：{self.summary}"
            })
        
        # 加入最近的对话
        messages.extend(self.recent_messages)
        
        return messages
    
    def chat(self, user_text):
        """处理一轮对话"""
        # 第 1 步：加入用户消息
        self.recent_messages.append({"role": "user", "content": user_text})
        
        # 第 2 步：检查是否需要压缩记忆
        # 每轮有 user + assistant 两条消息，所以超过 window_size*2 时压缩
        max_messages = self.window_size * 2
        if len(self.recent_messages) > max_messages:
            # 取出最早的 2 条消息（1 轮对话）
            old_msgs = self.recent_messages[:2]
            self.recent_messages = self.recent_messages[2:]
            
            # 生成摘要
            new_summary = self._summarize(old_msgs)
            if self.summary:
                self.summary += "\n" + new_summary
            else:
                self.summary = new_summary
            print(f"[记忆压缩] 已压缩早期对话，当前摘要：{self.summary}")
        
        # 第 3 步：构建 messages 并调用模型
        messages = self._build_messages()
        response = chat(messages)
        
        # 第 4 步：加入助手回复
        self.recent_messages.append({"role": "assistant", "content": response})
        
        return response
    
    def show_memory(self):
        """查看当前记忆状态"""
        print(f"摘要: {self.summary if self.summary else '(无)'}")
        print(f"最近对话: {len(self.recent_messages)} 条消息")
        for msg in self.recent_messages:
            print(f"  {msg['role']}: {msg['content'][:50]}")


# ===== 测试 =====
agent = SmartMemoryAgent(
    system_prompt="你是一个友好的中文客服，请用简短的话回答。",
    window_size=3  # 只保留最近 3 轮对话
)

print("=== 第 1 轮 ===")
print(f"助手: {agent.chat('我叫张三，订单号是 12345')}")

print("\n=== 第 2 轮 ===")
print(f"助手: {agent.chat('我的包裹什么时候到？')}")

print("\n=== 第 3 轮 ===")
print(f"助手: {agent.chat('能帮我查一下物流状态吗？')}")

print("\n=== 第 4 轮（触发压缩） ===")
print(f"助手: {agent.chat('另外我想改一下收货地址')}")

print("\n=== 第 5 轮（测试是否还记得） ===")
print(f"助手: {agent.chat('我的订单号是多少？')}")

print("\n=== 查看记忆状态 ===")
agent.show_memory()

### 4.3 运行分析

运行上面的代码，观察输出：

1. **前 3 轮**：正常对话，所有历史都在 `recent_messages` 中
2. **第 4 轮**：对话超过窗口大小（3轮），触发压缩
   - 最早的 1 轮对话被压缩成摘要
   - `recent_messages` 只保留最近 3 轮
3. **第 5 轮**：问"订单号是多少"
   - 模型通过**摘要**知道订单号是 12345
   - 虽然原始对话已经被"遗忘"，但关键信息保留在摘要中

**两种策略的对比：**

| | 示例一（全量记忆） | 示例二（滑动窗口+摘要） |
|---|---|---|
| 记忆方式 | 保存所有对话 | 只保留最近 N 轮 + 摘要 |
| token 消耗 | 随对话增长 | 基本稳定 |
| 信息保留 | 完整 | 近期完整，早期靠摘要 |
| 适用场景 | 短对话（<20轮） | 长对话（几十上百轮） |

---

## 五、课后练习

### 练习 1（基础）：多轮对话客服

**目标：** 实现一个能记住对话历史的客服智能体，验证它确实能"记住"之前说过的内容。

**要求：**
1. 补全 `CustomerServiceAgent` 类
2. 实现 `chat` 方法：把用户消息加入历史，拼接完整上下文发给模型
3. 实现 `reset` 方法：清空记忆，开始新对话
4. 实现 `show_history` 方法：打印当前对话历史

**提示：**
- 参考示例一的 `SimpleMemoryAgent`
- `messages` 列表的第一个元素应该是 system prompt
- 每次调用 API 时，把完整的 messages 列表传给模型

**测试用例：**
- 第 1 轮："我想买一台笔记本电脑"
- 第 2 轮："预算大概 5000 元左右"
- 第 3 轮："你刚才说我的预算是多少？"（应该能回答 5000 元）
- 调用 `reset()` 后，再问"我的预算是多少？"（应该不知道了）

In [ ]:
# ===== 练习 1：多轮对话客服 =====

class CustomerServiceAgent:
    """多轮对话客服智能体"""
    
    def __init__(self):
        # TODO: 初始化消息历史，设置 system prompt
        # 提示：system prompt 可以设置为 "你是一个电商客服，需要了解用户需求后推荐合适的商品。"
        self.messages = []
    
    def chat(self, user_text):
        """处理一轮对话"""
        # TODO: 1. 把用户消息加入历史
        # TODO: 2. 调用 API 获取回复
        # TODO: 3. 把助手回复加入历史
        # TODO: 4. 返回回复
        pass
    
    def reset(self):
        """清空记忆，开始新对话"""
        # TODO: 重置消息历史（保留 system prompt）
        pass
    
    def show_history(self):
        """打印对话历史"""
        # TODO: 遍历 messages，打印每条消息的角色和内容
        pass


# ===== 测试 =====
agent = CustomerServiceAgent()

print("=== 第 1 轮 ===")
print(f"客服: {agent.chat('我想买一台笔记本电脑')}")

print("\n=== 第 2 轮 ===")
print(f"客服: {agent.chat('预算大概 5000 元左右')}")

print("\n=== 第 3 轮 ===")
print(f"客服: {agent.chat('你刚才说我的预算是多少？')}")

print("\n=== 查看历史 ===")
agent.show_history()

print("\n=== 重置记忆 ===")
agent.reset()
print(f"客服: {agent.chat('我的预算是多少？')}")

### 练习 2（进阶）：带记忆的工具调用智能体

**目标：** 结合项目二的 Tool Use 和本项目的 Memory，构建一个**既能记住对话、又能调用工具**的智能体。

**场景：** 一个电商客服，能记住用户之前说的话，同时能查询订单状态。

**要求：**
1. 实现 `check_order` 工具函数（模拟查询订单）
2. 编写工具的 JSON Schema 描述
3. 补全 `MemoryToolAgent` 类，实现：
   - 多轮对话记忆（messages 列表）
   - 工具调用处理（tool_calls 解析）
   - 滑动窗口（超过 N 轮时截断早期对话）

**提示：**
- 参考示例二的 `_build_messages` 方法构建 messages
- 参考项目二的 `weather_agent` 处理 tool_calls
- 注意：截断时不要截断 system prompt 和 tool 相关的消息

**测试用例：**
- 第 1 轮："我的订单号是 ORD-2024-001"
- 第 2 轮："帮我查一下物流状态"
- 第 3 轮："大概几天能到？"
- 第 4 轮："我的订单号是多少来着？"（应该能回答）

In [ ]:
import json

# ===== 练习 2：带记忆的工具调用智能体 =====

# 第一步：实现工具函数
ORDER_DB = {
    "ORD-2024-001": {"status": "已发货", "eta": "预计 3 天到达", "courier": "顺丰速运"},
    "ORD-2024-002": {"status": "待发货", "eta": "预计 5-7 天", "courier": "待分配"},
}

def check_order(order_id: str) -> str:
    """查询订单状态"""
    # TODO: 从 ORDER_DB 中查找订单，返回格式化的订单信息
    # 找不到时返回友好提示
    pass


# 第二步：编写工具描述（JSON Schema）
tools = [
    # TODO: 编写 check_order 工具的 JSON Schema
]


# 第三步：构建带记忆的工具调用智能体
class MemoryToolAgent:
    """带记忆 + 工具调用的智能体"""
    
    def __init__(self, window_size=5):
        self.window_size = window_size
        # TODO: 初始化 messages 列表（包含 system prompt 和工具描述）
        self.messages = []
    
    def chat(self, user_text):
        """处理一轮对话（含工具调用）"""
        # TODO: 1. 加入用户消息
        # TODO: 2. 调用 API（带 tools 参数）
        # TODO: 3. 如果有 tool_calls，执行工具并把结果加入 messages
        # TODO: 4. 再次调用 API 获取最终回复
        # TODO: 5. 检查是否需要截断早期对话
        # TODO: 6. 返回回复
        pass
    
    def _trim_memory(self):
        """滑动窗口：超过窗口大小时，截断早期对话"""
        # TODO: 保留 system prompt，截断最早的对话
        # 提示：注意不要截断 tool 相关的消息
        pass


# ===== 测试 =====
agent = MemoryToolAgent(window_size=4)

print("=== 第 1 轮 ===")
print(f"客服: {agent.chat('你好，我想查一下订单')}")

print("\n=== 第 2 轮 ===")
print(f"客服: {agent.chat('订单号是 ORD-2024-001')}")

print("\n=== 第 3 轮 ===")
print(f"客服: {agent.chat('大概几天能到？')}")

print("\n=== 第 4 轮 ===")
print(f"客服: {agent.chat('我的订单号是多少来着？')}")

print("\n=== 查看记忆 ===")
print(f"当前 messages 数量: {len(agent.messages)}")

---
## 小结

恭喜你完成了第三个项目！回顾一下核心知识点：

| 知识点 | 要点 |
|--------|------|
| 大模型为什么"失忆" | API 是无状态的，每次调用独立 |
| 短期记忆的本质 | 把对话历史存在 messages 列表中，每次调用时一起发给模型 |
| messages 的角色 | system（指令）、user（用户）、assistant（助手） |
| 滑动窗口 | 只保留最近 N 轮对话，避免 token 超限 |
| 摘要压缩 | 把早期对话总结成摘要，保留关键信息 |
| 记忆 + 工具 | Memory 和 Tool Use 可以组合使用，构建更强大的智能体 |

**核心流程：** 用户说话 → 加入历史 → 拼接上下文 → 调用模型 → 保存回复 → 循环

**下一步预告：** 在[项目四](./project4_code_sandbox.ipynb)中，我们将学习如何让智能体**自己写代码、自己执行**，实现类似 Code Interpreter 的功能！